### Chatbot And RAG Evaluation

Retrieval Augmented Generation (RAG) is a technique that enhances Large Language Models (LLMs) by providing them with relevant external knowledge. It has become one of the most widely used approaches for building LLM applications.

This tutorial will show you how to evaluate your RAG applications using LangSmith. You'll learn:

1. How to create test datasets
2. How to run your RAG application on those datasets
3. How to measure your application's performance using different evaluation metrics

#### Overview
A typical RAG evaluation workflow consists of three main steps:

1. Creating a dataset with questions and their expected answers
2. Running your RAG application on those questions
3. Using evaluators to measure how well your application performed, looking at factors like:
 - Answer relevance
 - Answer accuracy
 - Retrieval quality
 
For this tutorial, we'll create and evaluate a bot that answers questions about a few of Lilian Weng's insightful blog posts.

### Chatbot Evaluation

In [24]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGSMITH_API_KEY"]=os.getenv("LANGSMITH_API_KEY")
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ["LANGSMITH_TRACING"]="true"

In [25]:
from langsmith import Client

client = Client()

# Define dataset: these are your test cases
dataset_name = "Chatbots Evaluation Examples Query"
dataset = client.create_dataset(dataset_name)
client.create_examples(
    dataset_id=dataset.id,
    examples=[
        {
            "inputs": {"question": "What is LangChain?"},
            "outputs": {"answer": "A framework for building LLM applications"},
        },
        {
            "inputs": {"question": "What is LangSmith?"},
            "outputs": {"answer": "A platform for observing and evaluating LLM applications"},
        },
        {
            "inputs": {"question": "What is OpenAI?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        },
        {
            "inputs": {"question": "What is Google?"},
            "outputs": {"answer": "A technology company known for search"},
        },
        {
            "inputs": {"question": "What is Mistral?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        }
    ]
)

{'example_ids': ['34612b7f-4ec3-4ad1-90ba-75ffb8aac2e5',
  '1405ce1d-e676-4567-b20d-67eb46d66ecb',
  '4978b4fa-199d-4bc4-bc71-76db84acb299',
  '61b6eb5e-b497-4f3f-aab7-3a97088396e9',
  'b71ab136-da52-48b3-9e02-71bbea0d755a'],
 'count': 5,
 'as_of': '2026-06-04T20:29:50.312364486Z'}

### Define Metrics (LLM As A Judge)


In [26]:
from groq import Groq
import os

groq_client = Groq(
    api_key=os.getenv("GROQ_API_KEY")
)

eval_instructions = (
    "You are an expert professor specialized in grading "
    "students' answers to questions."
)

def correctness(
    inputs: dict,
    outputs: dict,
    reference_outputs: dict
) -> bool:

    user_content = f"""
You are grading the following question:
{inputs['question']}

Here is the real answer:
{reference_outputs['answer']}

You are grading the following predicted answer:
{outputs['response']}

Respond with only one word:
CORRECT or INCORRECT

Grade:
"""

    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",  # or llama-3.3-70b-versatile
        temperature=0,
        messages=[
            {
                "role": "system",
                "content": eval_instructions
            },
            {
                "role": "user",
                "content": user_content
            }
        ]
    )

    grade = response.choices[0].message.content.strip()

    return grade == "CORRECT"

In [27]:
## Concisions- checks whether the actual output is less than 2x the length of the expected result.
def concision(outputs: dict, reference_outputs: dict) -> bool:
    return int(len(outputs["response"]) < 2 * len(reference_outputs["answer"]))

### Run Evaluations

In [28]:
default_instructions = (
    "Respond to the users question in a short, concise manner "
    "(one short sentence)."
)

def my_app(
    question: str,
    model: str = "llama-3.1-8b-instant",
    instructions: str = default_instructions,
) -> str:

    response = groq_client.chat.completions.create(
        model=model,
        temperature=0,
        messages=[
            {
                "role": "system",
                "content": instructions
            },
            {
                "role": "user",
                "content": question
            }
        ]
    )

    return response.choices[0].message.content

In [29]:
### Call my_app for every datapoints
def ls_target(inputs: str) -> dict:
    return {"response": my_app(inputs["question"])}

In [30]:
experiment_results = client.evaluate(
    ls_target,
    data=dataset_name,
    evaluators=[
        correctness,
        concision
    ],
    experiment_prefix="groq-chatbot"
)

View the evaluation results for experiment: 'groq-chatbot-36a81aa0' at:
https://smith.langchain.com/o/8086bb00-9700-4e9f-b376-84537e0bf843/datasets/7052b94a-ef61-477b-a8c1-4c6eefa1d70b/compare?selectedSessions=c836ce35-88ab-4bcb-a56b-b73b90f92bad




5it [00:02,  1.89it/s]


In [31]:
def ls_target(inputs: dict) -> dict:
    return {
        "response": my_app(
            inputs["question"],
            model="llama-3.1-8b-instant"
        )
    }

In [32]:
## Run our evaluation
experiment_results=client.evaluate(
    ls_target, ## Your AI system
    data=dataset_name,
    evaluators=[correctness,concision],
    experiment_prefix="groq-chatbot"
)

View the evaluation results for experiment: 'groq-chatbot-db46f08e' at:
https://smith.langchain.com/o/8086bb00-9700-4e9f-b376-84537e0bf843/datasets/7052b94a-ef61-477b-a8c1-4c6eefa1d70b/compare?selectedSessions=4ad5beb7-7230-40b9-8ab6-0c4b00322c65




5it [00:02,  2.33it/s]


### Evaluation For RAG

In [33]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

urls = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
    "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/",
]

docs = [WebBaseLoader(url).load() for url in urls]
docs_list = [item for sublist in docs for item in sublist]

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=250,
    chunk_overlap=0
)

doc_splits = text_splitter.split_documents(docs_list)

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = InMemoryVectorStore.from_documents(
    documents=doc_splits,
    embedding=embeddings
)

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 6}
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1645.43it/s]


In [34]:
retriever.invoke("what is agents")

[Document(id='e620fec0-93ac-4f20-bf92-20078ee5a616', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'title': "LLM Powered Autonomous Agents | Lil'Log", 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview\nIn a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:\n\nPlanning\n\nSubgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.\nReflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from mistakes and refine them for future steps, 

In [35]:
from langchain.chat_models import init_chat_model
llm=init_chat_model("groq:llama-3.1-8b-instant")
llm

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x00000257A503B1D0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000257A4C6FCE0>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [36]:
from langsmith import traceable

## Add decorator
@traceable()
def rag_bot(question:str)->dict:
    ## Relevant context
    docs=retriever.invoke(question)
    docs_string = " ".join(doc.page_content for doc in docs)

    instructions = f"""You are a helpful assistant who is good at analyzing source information and answering questions.       Use the following source documents to answer the user's questions.       If you don't know the answer, just say that you don't know.       Use three sentences maximum and keep the answer concise.

Documents:
{docs_string}"""
    
    ## llm invoke

    ai_msg=llm.invoke([
         {"role": "system", "content": instructions},
        {"role": "user", "content": question},

    ])
    return {"answer":ai_msg.content,"documents":docs}

In [37]:
rag_bot("What is agents")

{'answer': 'In the context of LLM-powered autonomous agents, an agent refers to a software program that can perform tasks, interact with its environment, and make decisions based on its observations and planning. These agents are powered by large language models (LLMs) that serve as their "brain" and enable them to learn, reason, and behave in a human-like manner.',
 'documents': [Document(id='e620fec0-93ac-4f20-bf92-20078ee5a616', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'title': "LLM Powered Autonomous Agents | Lil'Log", 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview\nIn a LLM-powered autonomous agent system, LLM

### Dataset

In [39]:
from langsmith import Client

client=Client()

# Define the examples for the dataset
examples = [
    {
        "inputs": {"question": "How does the ReAct agent use self-reflection? "},
        "outputs": {"answer": "ReAct integrates reasoning and acting, performing actions - such tools like Wikipedia search API - and then observing / reasoning about the tool outputs."},
    },
    {
        "inputs": {"question": "What are the types of biases that can arise with few-shot prompting?"},
        "outputs": {"answer": "The biases that can arise with few-shot prompting include (1) Majority label bias, (2) Recency bias, and (3) Common token bias."},
    },
    {
        "inputs": {"question": "What are five types of adversarial attacks?"},
        "outputs": {"answer": "Five types of adversarial attacks are (1) Token manipulation, (2) Gradient based attack, (3) Jailbreak prompting, (4) Human red-teaming, (5) Model red-teaming."},
    }
]

### create the daatset and example in LAngsmith
dataset_name="RAG Test Evaluation 1"
dataset = client.create_dataset(dataset_name=dataset_name)
client.create_examples(
    dataset_id=dataset.id,
    examples=examples
)

{'example_ids': ['757b379c-731c-4668-b820-9423a9d9c48a',
  '272eeb89-c414-45cd-9173-02af355461d2',
  '076ea8ff-3dcb-4f70-a8fd-e7498fdc78ad'],
 'count': 3,
 'as_of': '2026-06-04T20:42:21.712699266Z'}

### Evaluators or Metrics
1. Correctness: Response vs reference answer
- Goal: Measure "how similar/correct is the RAG chain answer, relative to a ground-truth answer"
- Mode: Requires a ground truth (reference) answer supplied through a dataset
- Evaluator: Use LLM-as-judge to assess answer correctness.

In [40]:
"""
LangSmith Correctness Evaluator using Groq + Structured Output

Purpose:
--------
Evaluates whether a generated answer is factually correct relative
to a reference answer.

Workflow:
---------
Question
    ↓
Reference Answer
    ↓
Generated Answer
    ↓
Groq Judge LLM
    ↓
Structured Output
    ↓
True / False

Requirements:
-------------
pip install langchain-groq
pip install langchain
"""

from typing_extensions import TypedDict, Annotated

from langchain_groq import ChatGroq


# ------------------------------------------------------------------
# Structured Output Schema
# ------------------------------------------------------------------

class CorrectnessGrade(TypedDict):
    """
    Structured schema returned by the evaluator model.

    Attributes:
        explanation:
            Step-by-step reasoning used by the judge model.

        correct:
            True if the generated answer is factually correct,
            False otherwise.
    """

    explanation: Annotated[
        str,
        ...,
        "Explain your reasoning for the score."
    ]

    correct: Annotated[
        bool,
        ...,
        "True if the answer is correct, False otherwise."
    ]


# ------------------------------------------------------------------
# Evaluation Instructions
# ------------------------------------------------------------------

correctness_instructions = """
You are a teacher grading a quiz.

You will be given:

1. QUESTION
2. GROUND TRUTH ANSWER
3. STUDENT ANSWER

Grade according to the following rules:

(1) Grade ONLY based on factual accuracy relative to the ground truth.
(2) The student answer must not contain conflicting information.
(3) Additional information is allowed if it is factually correct.

Correctness Rules:

- correct=True if all criteria are satisfied.
- correct=False if any criterion is violated.

Think step-by-step before deciding.

Do not simply restate the answer.
"""


# ------------------------------------------------------------------
# Judge LLM
# ------------------------------------------------------------------

grader_llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
).with_structured_output(
    CorrectnessGrade
)


# ------------------------------------------------------------------
# LangSmith Evaluator
# ------------------------------------------------------------------

def correctness(
    inputs: dict,
    outputs: dict,
    reference_outputs: dict
) -> bool:
    """
    LangSmith evaluator that checks factual correctness.

    Args:
        inputs:
            Dataset input.
            Example:
            {
                "question": "What is the capital of France?"
            }

        outputs:
            Application output.
            Example:
            {
                "answer": "Paris"
            }

        reference_outputs:
            Ground truth output.
            Example:
            {
                "answer": "Paris"
            }

    Returns:
        bool:
            True if the generated answer is correct,
            False otherwise.
    """

    evaluation_prompt = f"""
QUESTION:
{inputs['question']}

GROUND TRUTH ANSWER:
{reference_outputs['answer']}

STUDENT ANSWER:
{outputs['answer']}
"""

    grade = grader_llm.invoke(
        [
            {
                "role": "system",
                "content": correctness_instructions
            },
            {
                "role": "user",
                "content": evaluation_prompt
            }
        ]
    )

    return grade["correct"]

experiment_results = client.evaluate(
    ls_target,
    data=dataset_name,
    evaluators=[correctness],
    experiment_prefix="groq-rag-correctness"
)

View the evaluation results for experiment: 'groq-rag-correctness-d674cec5' at:
https://smith.langchain.com/o/8086bb00-9700-4e9f-b376-84537e0bf843/datasets/e3836600-d86b-4a72-9fc3-b96aff3f41ec/compare?selectedSessions=e7f04906-d845-4b38-b683-a879aaea10a4




0it [00:00, ?it/s]Error running evaluator <DynamicRunEvaluator correctness> on run 019e945f-8411-7bc3-a2f5-f830b86196cd: KeyError('answer')
Traceback (most recent call last):
  File "d:\GenAI\RAGTutorial\.venv\Lib\site-packages\langsmith\evaluation\_runner.py", line 1637, in _run_evaluators
    evaluator_response = evaluator.evaluate_run(  # type: ignore[call-arg]
        run=run,
        example=example,
        evaluator_run_id=evaluator_run_id,
    )
  File "d:\GenAI\RAGTutorial\.venv\Lib\site-packages\langsmith\evaluation\evaluator.py", line 332, in evaluate_run
    result = self.func(
        run,
        example,
        langsmith_extra={"run_id": evaluator_run_id, "metadata": metadata},
    )
  File "d:\GenAI\RAGTutorial\.venv\Lib\site-packages\langsmith\run_helpers.py", line 777, in wrapper
    function_result = run_container["context"].run(
        func, *args, **kwargs
    )
  File "d:\GenAI\RAGTutorial\.venv\Lib\site-packages\langsmith\evaluation\evaluator.py", line 758, in 

### Relevance: Response vs input
The flow is similar to above, but we simply look at the inputs and outputs without needing the reference_outputs. Without a reference answer we can't grade accuracy, but can still grade relevance—as in, did the model address the user's question or not.

In [41]:
"""
LangSmith Relevance Evaluator using Groq + Structured Output

Purpose:
--------
Evaluates whether a generated answer is relevant to the user's question.

Workflow:
---------
Question
    ↓
Generated Answer
    ↓
Groq Judge LLM
    ↓
Structured Output
    ↓
True / False

Requirements:
-------------
pip install langchain-groq
pip install langchain
"""

from typing_extensions import TypedDict, Annotated

from langchain_groq import ChatGroq


# ------------------------------------------------------------------
# Structured Output Schema
# ------------------------------------------------------------------

class RelevanceGrade(TypedDict):
    """
    Structured schema returned by the evaluator model.

    Attributes:
        explanation:
            Step-by-step reasoning used by the judge model.

        relevant:
            True if the answer is relevant to the question,
            False otherwise.
    """

    explanation: Annotated[
        str,
        ...,
        "Explain your reasoning for the score."
    ]

    relevant: Annotated[
        bool,
        ...,
        "True if the answer addresses the question."
    ]


# ------------------------------------------------------------------
# Evaluation Instructions
# ------------------------------------------------------------------

relevance_instructions = """
You are a teacher grading a quiz.

You will be given:

1. QUESTION
2. STUDENT ANSWER

Grade according to the following rules:

(1) Ensure the STUDENT ANSWER is concise and relevant to the QUESTION.
(2) Ensure the STUDENT ANSWER helps answer the QUESTION.

Relevance Rules:

- relevant=True if all criteria are satisfied.
- relevant=False if any criterion is violated.

Think step-by-step before deciding.

Do not simply restate the answer.
"""


# ------------------------------------------------------------------
# Judge LLM
# ------------------------------------------------------------------

relevance_llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
).with_structured_output(
    RelevanceGrade
)


# ------------------------------------------------------------------
# LangSmith Evaluator
# ------------------------------------------------------------------

def relevance(
    inputs: dict,
    outputs: dict
) -> bool:
    """
    LangSmith evaluator that checks answer relevance.

    Args:
        inputs:
            Dataset input.
            Example:
            {
                "question": "What is RAG?"
            }

        outputs:
            Application output.
            Example:
            {
                "answer": "RAG combines retrieval and generation."
            }

    Returns:
        bool:
            True if the answer is relevant to the question,
            False otherwise.
    """

    evaluation_prompt = f"""
QUESTION:
{inputs['question']}

STUDENT ANSWER:
{outputs['answer']}
"""

    grade = relevance_llm.invoke(
        [
            {
                "role": "system",
                "content": relevance_instructions
            },
            {
                "role": "user",
                "content": evaluation_prompt
            }
        ]
    )

    return grade["relevant"]

### Groundedness: Response vs retrieved docs
Another useful way to evaluate responses without needing reference answers is to check if the response is justified by (or "grounded in") the retrieved documents.

In [42]:
"""
LangSmith Groundedness Evaluator using Groq + Structured Output

Purpose:
--------
Evaluates whether a generated answer is grounded in
the retrieved documents and does not hallucinate.

Workflow:
---------
Retrieved Documents
        ↓
Generated Answer
        ↓
Groq Judge LLM
        ↓
Structured Output
        ↓
True / False

Requirements:
-------------
pip install langchain-groq
pip install langchain
"""

from typing_extensions import TypedDict, Annotated

from langchain_groq import ChatGroq


# ------------------------------------------------------------------
# Structured Output Schema
# ------------------------------------------------------------------

class GroundedGrade(TypedDict):
    """
    Structured schema returned by the evaluator model.

    Attributes:
        explanation:
            Step-by-step reasoning produced by the judge.

        grounded:
            True if the answer is fully supported
            by the retrieved documents.
    """

    explanation: Annotated[
        str,
        ...,
        "Explain your reasoning for the score."
    ]

    grounded: Annotated[
        bool,
        ...,
        "True if the answer is grounded in the retrieved documents."
    ]


# ------------------------------------------------------------------
# Evaluation Instructions
# ------------------------------------------------------------------

grounded_instructions = """
You are a teacher grading a quiz.

You will be given:

1. FACTS
2. STUDENT ANSWER

Grade according to the following rules:

(1) Ensure the STUDENT ANSWER is grounded in the FACTS.
(2) Ensure the STUDENT ANSWER does not contain hallucinated information.
(3) All factual claims must be supported by the FACTS.

Groundedness Rules:

- grounded=True if all criteria are satisfied.
- grounded=False if any unsupported claim exists.

Think step-by-step before deciding.

Do not simply restate the answer.
"""


# ------------------------------------------------------------------
# Judge LLM
# ------------------------------------------------------------------

grounded_llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
).with_structured_output(
    GroundedGrade
)


# ------------------------------------------------------------------
# LangSmith Evaluator
# ------------------------------------------------------------------

def groundedness(
    inputs: dict,
    outputs: dict
) -> bool:
    """
    LangSmith evaluator that checks whether the
    generated answer is grounded in the retrieved
    documents.

    Args:
        inputs:
            Dataset input.
            Example:
            {
                "question": "What is RAG?"
            }

        outputs:
            RAG system output.
            Example:
            {
                "answer": "...",
                "documents": [...]
            }

    Returns:
        bool:
            True if the answer is fully supported
            by the retrieved documents.
    """

    # Convert retrieved documents into one context string
    document_context = "\n\n".join(
        doc.page_content
        for doc in outputs["documents"]
    )

    evaluation_prompt = f"""
FACTS:
{document_context}

STUDENT ANSWER:
{outputs['answer']}
"""

    grade = grounded_llm.invoke(
        [
            {
                "role": "system",
                "content": grounded_instructions
            },
            {
                "role": "user",
                "content": evaluation_prompt
            }
        ]
    )

    return grade["grounded"]

### Retrieval Relevance: Retrieved docs vs input

In [43]:
"""
LangSmith Relevance Evaluator using Groq + Structured Output

Purpose:
--------
Evaluates whether a generated answer is relevant to the user's question.

Workflow:
---------
Question
    ↓
Generated Answer
    ↓
Groq Judge LLM
    ↓
Structured Output
    ↓
True / False

Requirements:
-------------
pip install langchain-groq
pip install langchain
"""

from typing_extensions import TypedDict, Annotated

from langchain_groq import ChatGroq


# ------------------------------------------------------------------
# Structured Output Schema
# ------------------------------------------------------------------

class RelevanceGrade(TypedDict):
    """
    Structured schema returned by the evaluator model.

    Attributes:
        explanation:
            Step-by-step reasoning used by the judge model.

        relevant:
            True if the answer is relevant to the question,
            False otherwise.
    """

    explanation: Annotated[
        str,
        ...,
        "Explain your reasoning for the score."
    ]

    relevant: Annotated[
        bool,
        ...,
        "True if the answer addresses the question."
    ]


# ------------------------------------------------------------------
# Evaluation Instructions
# ------------------------------------------------------------------

relevance_instructions = """
You are a teacher grading a quiz.

You will be given:

1. QUESTION
2. STUDENT ANSWER

Grade according to the following rules:

(1) Ensure the STUDENT ANSWER is concise and relevant to the QUESTION.
(2) Ensure the STUDENT ANSWER helps answer the QUESTION.

Relevance Rules:

- relevant=True if all criteria are satisfied.
- relevant=False if any criterion is violated.

Think step-by-step before deciding.

Do not simply restate the answer.
"""


# ------------------------------------------------------------------
# Judge LLM
# ------------------------------------------------------------------

relevance_llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
).with_structured_output(
    RelevanceGrade
)


# ------------------------------------------------------------------
# LangSmith Evaluator
# ------------------------------------------------------------------

def relevance(
    inputs: dict,
    outputs: dict
) -> bool:
    """
    LangSmith evaluator that checks answer relevance.

    Args:
        inputs:
            Dataset input.
            Example:
            {
                "question": "What is RAG?"
            }

        outputs:
            Application output.
            Example:
            {
                "answer": "RAG combines retrieval and generation."
            }

    Returns:
        bool:
            True if the answer is relevant to the question,
            False otherwise.
    """

    evaluation_prompt = f"""
QUESTION:
{inputs['question']}

STUDENT ANSWER:
{outputs['answer']}
"""

    grade = relevance_llm.invoke(
        [
            {
                "role": "system",
                "content": relevance_instructions
            },
            {
                "role": "user",
                "content": evaluation_prompt
            }
        ]
    )

    return grade["relevant"]

### Run the evaluation

In [46]:
def target(inputs: dict) -> dict:
    """
    LangSmith target function.

    Executes the RAG application for each
    dataset example and returns the output
    to be evaluated.
    """

    return rag_bot(inputs["question"])


experiment_results = client.evaluate(
    target,
    data=dataset_name,
    evaluators=[
        correctness,
        groundedness,
        relevance,
        # retrieval_relevance
    ],
    experiment_prefix="groq-rag-evaluation",
    metadata={
        "model": "llama-3.3-70b-versatile",
        "framework": "LangChain",
        "architecture": "RAG"
    }
)

# Convert results to DataFrame
results_df = experiment_results.to_pandas()

print(results_df.head())

View the evaluation results for experiment: 'groq-rag-evaluation-f7b86769' at:
https://smith.langchain.com/o/8086bb00-9700-4e9f-b376-84537e0bf843/datasets/e3836600-d86b-4a72-9fc3-b96aff3f41ec/compare?selectedSessions=383df612-ef11-4b36-9de5-8311e42d85b1




3it [00:07,  2.46s/it]


                                     inputs.question  \
0        What are five types of adversarial attacks?   
1  What are the types of biases that can arise wi...   
2     How does the ReAct agent use self-reflection?    

                                      outputs.answer  \
0  According to the provided source, the five typ...   
1  According to Zhao et al. (2021), few-shot prom...   
2  ReAct uses self-reflection by providing the LL...   

                                   outputs.documents error  \
0  [page_content='Adversarial attacks are inputs ...  None   
1  [page_content='Text: i'll bet the video game i...  None   
2  [page_content='Self-reflection is a vital aspe...  None   

                                    reference.answer  feedback.correctness  \
0  Five types of adversarial attacks are (1) Toke...                  True   
1  The biases that can arise with few-shot prompt...                  True   
2  ReAct integrates reasoning and acting, perform...               